# 01 - Extracción de Datos
## Sistema de Predicción Temprana de Plagas - Sierra del Patlachique

**Objetivo:** Descargar datos climáticos y edáficos de la API NASA POWER

**Responsabilidades:**
- Conectar a API NASA POWER
- Descargar parámetros
- Transformación mínima (renombrar columnas)
- Exportar datos crudos

## Ubicación y Periodo

**Zona de Estudio:** Sierra del Patlachique, Estado de México
- Latitud: 19.65°N
- Longitud: -98.85°W
- Altitud: ~2,500 msnm
- Suelos: Phaeozems y Vertisoles (arcillosos)

In [1]:
import pandas as pd
import requests
from datetime import datetime
import os

## 1. Configuración de Parámetros

In [2]:
# Ubicación del sitio de estudio
LAT = 19.65
LON = -98.85
LOCATION_NAME = "Sierra del Patlachique, Edomex"

# Rango temporal
FECHA_INICIO = "20240101"
FECHA_FIN = datetime.now().strftime("%Y%m%d")

# Parámetros a descargar de NASA POWER
PARAMETROS = {
    "T2M": "Temperatura media a 2m (°C)",
    "PRECTOTCORR": "Precipitación total corregida (mm)",
    "GWETTOP": "Humedad relativa del suelo - capa superficial (0-1)",
    "RH2M": "Humedad relativa a 2m (%)"
}

PARAMS_STRING = ",".join(PARAMETROS.keys())

print(f"Ubicación: {LOCATION_NAME}")
print(f"Período: {FECHA_INICIO} a {FECHA_FIN}")
print(f"\nParámetros a descargar:")
for param, desc in PARAMETROS.items():
    print(f"\t{param}: {desc}")

Ubicación: Sierra del Patlachique, Edomex
Período: 20240101 a 20260327

Parámetros a descargar:
	T2M: Temperatura media a 2m (°C)
	PRECTOTCORR: Precipitación total corregida (mm)
	GWETTOP: Humedad relativa del suelo - capa superficial (0-1)
	RH2M: Humedad relativa a 2m (%)


## 2. Conexión y Descarga de API NASA POWER

In [3]:
# Construcción de URL
url = (
    f"https://power.larc.nasa.gov/api/temporal/daily/point?"
    f"parameters={PARAMS_STRING}"
    f"&community=AG"
    f"&longitude={LON}"
    f"&latitude={LAT}"
    f"&start={FECHA_INICIO}"
    f"&end={FECHA_FIN}"
    f"&format=JSON"
)

print("Conectando con API NASA POWER...")
print(f"URL: {url[:80]}...\n")

try:
    response = requests.get(url)
    response.raise_for_status()
    raw_data = response.json()['properties']['parameter']
    print(f"✓ Descarga exitosa")
    print(f"  - Datos descargados: {len(raw_data)} registros")
except Exception as e:
    print(f"✗ Error en la descarga: {e}")
    raise

Conectando con API NASA POWER...
URL: https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M,PRECTOTCORR,...

✓ Descarga exitosa
  - Datos descargados: 4 registros


## 3. Transformación Mínima: Renombrado de Columnas

In [4]:
# Crear DataFrame desde datos crudos
df_patlachique = pd.DataFrame(raw_data)

# Convertir índice a datetime
df_patlachique.index = pd.to_datetime(df_patlachique.index, format='%Y%m%d')
df_patlachique.index.name = 'fecha'

# Mapeo de parámetros NASA a nombres descriptivos
rename_dict = {
    'T2M': 'temp_media_C',
    'PRECTOTCORR': 'lluvia_mm',
    'GWETTOP': 'humedad_suelo_frac',
    'RH2M': 'humedad_relativa_pct'
}

df_patlachique.rename(columns=rename_dict, inplace=True)

print(f"Datos transformados")
print(f"\t- Columnas: {list(df_patlachique.columns)}")
print(f"\t- Período: {df_patlachique.index.min().date()} a {df_patlachique.index.max().date()}")
print(f"\t- Registros: {len(df_patlachique)}")

Datos transformados
	- Columnas: ['temp_media_C', 'lluvia_mm', 'humedad_suelo_frac', 'humedad_relativa_pct']
	- Período: 2024-01-01 a 2026-03-27
	- Registros: 817


## 4. Validación Básica de Integridad

In [5]:
print("\nAUDITORÍA DE INTEGRIDAD - DATOS CRUDOS\n")

# 4.1 Verificar nulos
nulos = df_patlachique.isnull().sum()
print("Valores nulos por columna:")
print(nulos)
if nulos.sum() > 0:
    print(f"\t{nulos.sum()} valores nulos detectados (serán tratados en notebook 03)")
else:
    print("\tSin valores nulos")

# 4.2 Estadísticas descriptivas crudas
print("\nEstadísticas descriptivas (SIN LIMPIAR):")
print(df_patlachique.describe().round(2))

# 4.3 Detectar potenciales anomalías (reportar, no corregir)
print("\nPotenciales anomalías detectadas (no se corrigen aquí):")

# Temperatura: rangos físicos imposibles
temp_outliers = (
        (df_patlachique['temp_media_C'] > 45) | (df_patlachique['temp_media_C'] < -10)
).sum()
if temp_outliers > 0:
    print(f"\tTemperatura fuera de rango [-10°C, 45°C]: {temp_outliers} registros")

# Lluvia: valores negativos imposibles
lluvia_neg = (df_patlachique['lluvia_mm'] < 0).sum()
if lluvia_neg > 0:
    print(f"\tLluvia negativa: {lluvia_neg} registros")

# Humedad relativa: rango 0-100
hr_outliers = (
        (df_patlachique['humedad_relativa_pct'] > 100) | (df_patlachique['humedad_relativa_pct'] < 0)
).sum()
if hr_outliers > 0:
    print(f"\tHumedad relativa fuera de rango [0%, 100%]: {hr_outliers} registros")

# Humedad suelo: rango 0-1
hs_outliers = (
        (df_patlachique['humedad_suelo_frac'] > 1) | (df_patlachique['humedad_suelo_frac'] < 0)
).sum()
if hs_outliers > 0:
    print(f"\tHumedad suelo fuera de rango [0, 1]: {hs_outliers} registros")

if temp_outliers == 0 and lluvia_neg == 0 and hr_outliers == 0 and hs_outliers == 0:
    print("\tSin anomalías detectadas")

print("\n\tNota: Las anomalías se tratarán en notebook 03_data_cleaning.ipynb")


AUDITORÍA DE INTEGRIDAD - DATOS CRUDOS

Valores nulos por columna:
temp_media_C            0
lluvia_mm               0
humedad_suelo_frac      0
humedad_relativa_pct    0
dtype: int64
	Sin valores nulos

Estadísticas descriptivas (SIN LIMPIAR):
       temp_media_C  lluvia_mm  humedad_suelo_frac  humedad_relativa_pct
count        817.00     817.00              817.00                817.00
mean          10.86      -1.06               -3.15                 60.24
std           61.41      60.77               60.49                 66.65
min         -999.00    -999.00             -999.00               -999.00
25%           12.63       0.02                0.39                 49.24
50%           14.43       0.47                0.51                 68.44
75%           16.07       3.71                0.67                 79.06
max           22.72      35.93                0.82                 92.61

Potenciales anomalías detectadas (no se corrigen aquí):
	Temperatura fuera de rango [-10°C, 45°C

## 5. Vista Previa de Datos

In [6]:
print("Primeras 5 filas:")
print(df_patlachique.head())
print("\nÚltimas 5 filas:")
print(df_patlachique.tail())

Primeras 5 filas:
            temp_media_C  lluvia_mm  humedad_suelo_frac  humedad_relativa_pct
fecha                                                                        
2024-01-01         10.99       0.00                0.45                 55.47
2024-01-02         10.87       0.00                0.45                 58.37
2024-01-03         11.45       0.19                0.44                 52.55
2024-01-04         10.84       0.04                0.44                 70.67
2024-01-05         10.86       0.01                0.44                 68.84

Últimas 5 filas:
            temp_media_C  lluvia_mm  humedad_suelo_frac  humedad_relativa_pct
fecha                                                                        
2026-03-23         13.05       2.27                0.46                 64.93
2026-03-24         13.08       0.46                0.48                 65.38
2026-03-25       -999.00    -999.00             -999.00               -999.00
2026-03-26       -999.00    

## 6. Exportar Datos Crudos

In [7]:
# Crear directorio si no existe
os.makedirs("../data/raw", exist_ok=True)

# Ruta de guardado
output_path = "../data/raw/datos_patlachique_raw.csv"

# Guardar con índice de fecha
df_patlachique.to_csv(output_path, index=True, encoding='utf-8')

print(f"\nDatos crudos exportados")
print(f"\tRuta: {output_path}")
print(f"\tTamaño: {df_patlachique.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"\tRegistros: {len(df_patlachique)}")
print(f"\tPeríodo: {df_patlachique.index.min().date()} a {df_patlachique.index.max().date()}")


Datos crudos exportados
	Ruta: ../data/raw/datos_patlachique_raw.csv
	Tamaño: 31.91 KB
	Registros: 817
	Período: 2024-01-01 a 2026-03-27


## Resumen

* Datos descargados exitosamente de NASA POWER
* Renombrado de columnas para mayor claridad
* Validación básica de integridad realizada
* Exportados a `data/raw/`